# Knowledge Workflow — V5

Provider-agnostic two-stage knowledge extraction pipeline.

| Phase | What happens |
|---|---|
| **1 — Extraction** | LLM reads each abstract → pulls canonical + paper-specific concept pairs |
| **Normalization** | LLM deduplicates 100+ raw labels → clean 30-80 ontology-ready concepts |
| **2 — Schema** | LLM reads each full PDF → fills a wide-format schema (value + quote per concept) |

Outputs saved to `outputs/<collection>/`.

## 1 · Imports & Configuration

In [ ]:
from pyzotero import Zotero
from openai import OpenAI
import pandas as pd
from pypdf import PdfReader
from io import BytesIO
from datetime import datetime
from dotenv import load_dotenv
import json, re, glob, os, time

load_dotenv()
print('Imports OK')

In [ ]:
# ── Zotero ──────────────────────────────────────────────────────────────────
COLLECTION_ID       = 'VWMCLGL5'          # ← change to your collection key
ZOTERO_LIBRARY_ID   = '2189702'
ZOTERO_LIBRARY_TYPE = 'group'
ZOTERO_API_KEY      = os.getenv('ZOTERO_API_KEY', '')

# ── LLM provider (any OpenAI-compatible endpoint) ───────────────────────────
#   Provider        BASE_URL                               MODEL
#   OpenAI          https://api.openai.com/v1              gpt-4o
#   Anthropic       https://api.anthropic.com/v1           claude-sonnet-4-6
#   Groq            https://api.groq.com/openai/v1         llama-3.3-70b-versatile
#   Ollama (local)  http://localhost:11434/v1               llama3.2
LLM_BASE_URL = os.getenv('LLM_BASE_URL', 'https://api.anthropic.com/v1')
LLM_API_KEY  = os.getenv('LLM_API_KEY',  os.getenv('OPENAI_API_KEY', os.getenv('ANTHROPIC_API_KEY', '')))
MODEL        = os.getenv('LLM_MODEL',    'claude-sonnet-4-6')

FORCE_TOOL_CHOICE    = True
RATE_LIMIT_DELAY     = 0.5    # seconds between LLM calls
TOP_N_PER_PAPER      = 25     # concepts to extract per paper
FULL_TEXT_MAX_CHARS  = 80_000 # ~20k tokens

# Optional: skip Phase 1 and reuse an existing concept list
USE_CSV_CONCEPTS  = False
CONCEPTS_CSV_PATH = ''
CONCEPTS_COLUMN   = 'concept'

print(f'Provider : {LLM_BASE_URL}')
print(f'Model    : {MODEL}')

In [ ]:
zot    = Zotero(ZOTERO_LIBRARY_ID, ZOTERO_LIBRARY_TYPE, ZOTERO_API_KEY)
client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)
print('Zotero + LLM clients ready')

## 2 · Tool Schemas (OpenAI function-calling format)

In [ ]:
_EXTRACT_TOOL = {
    'type': 'function',
    'function': {
        'name': 'return_concepts',
        'description': (
            'Return key concepts extracted from the abstract. '
            'Each concept has a canonical ontology label AND the paper-specific term.'
        ),
        'parameters': {
            'type': 'object',
            'properties': {
                'concepts': {
                    'type': 'array',
                    'items': {
                        'type': 'object',
                        'properties': {
                            'canonical': {
                                'type': 'string',
                                'description': (
                                    'Ontology-ready label, lowercase, 1-4 words, suitable as a '
                                    'reusable column header. '
                                    'E.g. "absorber material", "device efficiency", "carrier lifetime".'
                                ),
                            },
                            'paper_term': {
                                'type': 'string',
                                'description': 'The specific term this paper uses for that concept.',
                            },
                            'relevance': {
                                'type': 'number',
                                'description': 'Relevance score 0.0–1.0',
                            },
                        },
                        'required': ['canonical', 'paper_term', 'relevance'],
                    },
                }
            },
            'required': ['concepts'],
        },
    },
}

_NORMALIZE_TOOL = {
    'type': 'function',
    'function': {
        'name': 'return_normalized_concepts',
        'description': 'Return a deduplicated, normalized list of ontology-ready concept labels.',
        'parameters': {
            'type': 'object',
            'properties': {
                'concepts': {
                    'type': 'array',
                    'items': {
                        'type': 'string',
                        'description': 'Canonical concept label: lowercase, 1-4 words.',
                    },
                }
            },
            'required': ['concepts'],
        },
    },
}

_SCHEMA_TOOL = {
    'type': 'function',
    'function': {
        'name': 'return_schema_values',
        'description': (
            'For each canonical concept, return the paper-specific value and '
            'the most informative source sentence from the paper text.'
        ),
        'parameters': {
            'type': 'object',
            'properties': {
                'values': {
                    'type': 'array',
                    'items': {
                        'type': 'object',
                        'properties': {
                            'canonical': {'type': 'string'},
                            'value':     {'type': 'string', 'description': 'Paper-specific term/measurement. Empty if not found.'},
                            'quote':     {'type': 'string', 'description': 'Most informative source sentence. Empty if not found.'},
                        },
                        'required': ['canonical', 'value', 'quote'],
                    },
                }
            },
            'required': ['values'],
        },
    },
}

print('Tool schemas defined')

## 3 · System Prompts

In [ ]:
_EXTRACT_SYSTEM = (
    'You are a scientific literature analyst and ontologist specialising in '
    'materials science and solar cell research. '
    'Given a paper abstract, extract domain-specific concepts in TWO forms:\n'
    '1. canonical — a general ontology-ready label (lowercase, 1-4 words) that '
    'could serve as a reusable column header across many papers in the field. '
    'Good examples: "absorber material", "device efficiency", "dopant species", '
    '"carrier lifetime", "passivation method", "open circuit voltage". '
    'Bad examples: "CdSeTe" (too specific), "22.1%" (a value, not a concept), '
    '"cell" (too vague).\n'
    '2. paper_term — the specific term, compound, percentage, or phrase this '
    'particular paper uses for that concept.\n'
    'Focus on technical properties, materials, methods, and performance metrics. '
    "Score each 0.0–1.0 by centrality to the paper's contribution."
)

_NORMALIZE_SYSTEM = (
    'You are a knowledge-graph ontologist. '
    'You will receive a list of candidate concept labels extracted by AI from '
    'multiple scientific papers in the solar cell materials domain. '
    'Your task: return a clean, deduplicated, normalized set of ontology-ready '
    'labels suitable as column headers in a knowledge-graph schema.\n\n'
    'Rules:\n'
    '- Merge near-synonyms into one canonical form '
    '(e.g. "open circuit voltage", "open-circuit voltage voc", "voc" → '
    '"open circuit voltage").\n'
    '- Keep labels lowercase, 1-4 words, general and reusable.\n'
    '- Remove labels that are too vague ("cell", "material"), too specific '
    '("CdSeTe", "22%"), or duplicates.\n'
    '- Aim for 30–80 high-quality, distinct concepts covering the corpus.\n'
    '- Order them roughly by domain importance (most central properties first).'
)

_SCHEMA_SYSTEM = (
    'You are a precise scientific data extractor. '
    'You will be given the full text of a scientific paper and a list of '
    'ontology concept labels. '
    'For EACH concept, find and return:\n'
    "  value  — the exact term, number, or very short phrase this paper uses "
    "for that concept (use the paper's own wording). "
    'If the concept is not addressed in this paper, use an empty string.\n'
    '  quote  — the single most informative sentence from the text that '
    'establishes or describes this concept. '
    'Prefer results sections, abstracts, or conclusion sentences. '
    'If not found, use an empty string.\n\n'
    'Be precise. Do not paraphrase. Do not invent values not in the text.'
)

print('System prompts defined')

## 4 · Utilities

In [ ]:
def make_filename(collection_name, username='Brent_Thompson', version=5):
    date = datetime.now().strftime('%Y%m%d')
    name = collection_name.replace(' ', '_').lower().replace('>', '')
    return f"{name}-{username}-v{version}-{date}.csv"


def find_latest_file(pattern):
    files = glob.glob(pattern)
    return max(files, key=os.path.getmtime) if files else None


def _tool_choice(name):
    if not FORCE_TOOL_CHOICE:
        return 'auto'
    return {'type': 'function', 'function': {'name': name}}


def _parse_tool_call(response, expected_name):
    msg = response.choices[0].message
    if not msg.tool_calls:
        return None
    for tc in msg.tool_calls:
        if tc.function.name == expected_name:
            try:
                return json.loads(tc.function.arguments)
            except json.JSONDecodeError:
                return None
    try:
        return json.loads(msg.tool_calls[0].function.arguments)
    except json.JSONDecodeError:
        return None


print('Utilities defined')

## 5 · Zotero Functions

In [ ]:
def get_collection_map(zot):
    """Return {'Parent > Subcollection': collection_id} for all collections."""
    all_collections = zot.everything(zot.collections())
    key_to_data     = {c['key']: c['data'] for c in all_collections}
    collection_map  = {}
    for c in all_collections:
        key        = c['key']
        name       = c['data']['name']
        parent_key = c['data'].get('parentCollection', False)
        path       = name
        while parent_key:
            parent_data = key_to_data.get(parent_key)
            if parent_data:
                path       = f"{parent_data['name']} > {path}"
                parent_key = parent_data.get('parentCollection', False)
            else:
                break
        collection_map[path] = key
    return collection_map


def get_pdf_text(item_key):
    """Extract full text from a Zotero item's PDF attachment."""
    for child in zot.children(item_key):
        if child['data'].get('contentType') == 'application/pdf':
            try:
                reader = PdfReader(BytesIO(zot.file(child['key'])))
                return ''.join(p.extract_text() or '' for p in reader.pages)
            except Exception:
                pass
    return ''


def get_collection_with_text(collection_id):
    """Return {title_lower: {metadata + full_text}} for all papers."""
    items      = zot.everything(zot.collection_items(collection_id))
    collection = {}
    for item in items:
        data = item['data']
        if data.get('itemType') in ('attachment', 'note') or not data.get('title'):
            continue
        collection[data['title'].lower()] = {
            'key':       item['key'],
            'title':     data['title'],
            'doi':       data.get('DOI', ''),
            'abstract':  data.get('abstractNote', ''),
            'date':      data.get('date', ''),
            'authors':   data.get('creators', []),
            'full_text': get_pdf_text(item['key']),
        }
    return collection


print('Zotero functions defined')

## 6 · Phase 1 — Concept Extraction

In [ ]:
def extract_concepts_from_abstract(abstract, top_n=25):
    """
    Extract concepts from a single abstract.
    Returns [{canonical, paper_term, relevance}] sorted by relevance desc.
    """
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=2048,
        messages=[
            {'role': 'system', 'content': _EXTRACT_SYSTEM},
            {'role': 'user',   'content': (
                f'Extract the top {top_n} concepts from this abstract. '
                f'For each, provide a canonical ontology label AND the paper-specific term.\n\n'
                f'Abstract:\n{abstract}'
            )},
        ],
        tools=[_EXTRACT_TOOL],
        tool_choice=_tool_choice('return_concepts'),
    )
    result = _parse_tool_call(response, 'return_concepts')
    if result:
        concepts = result.get('concepts', [])
        return sorted(concepts, key=lambda c: c.get('relevance', 0), reverse=True)[:top_n]
    return []


def build_concept_table(collection_dict, top_n=25):
    """
    Phase 1: run per-paper extraction across the collection.
    Returns (df_concepts, all_canonicals).
    """
    rows           = []
    all_canonicals = []
    papers         = [p for p in collection_dict.values() if p.get('abstract')]
    total          = len(papers)

    for i, paper in enumerate(papers, 1):
        print(f'  [{i}/{total}] Extracting: {paper["title"][:70]}')
        concepts = extract_concepts_from_abstract(paper['abstract'], top_n=top_n)
        for c in concepts:
            canon = c.get('canonical', '').strip().lower()
            rows.append({
                'paper':      paper['title'],
                'doi':        paper.get('doi', ''),
                'canonical':  canon,
                'paper_term': c.get('paper_term', ''),
                'relevance':  round(c.get('relevance', 0), 4),
            })
            all_canonicals.append(canon)
        if i < total:
            time.sleep(RATE_LIMIT_DELAY)

    return pd.DataFrame(rows), all_canonicals


print('Phase 1 functions defined')

## 7 · Normalization

In [ ]:
def normalize_concept_list(all_canonicals):
    """
    Deduplication pass: merges near-synonyms, returns 30–80 clean labels.
    """
    unique = sorted(set(c for c in all_canonicals if c))
    if not unique:
        return []

    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=2048,
        messages=[
            {'role': 'system', 'content': _NORMALIZE_SYSTEM},
            {'role': 'user',   'content': (
                f'Here are {len(unique)} candidate concept labels from a corpus of '
                f'domain-specific peer-reviewed research papers. '
                f'Normalize and deduplicate into a clean ontology-ready list (30–80 concepts).\n\n'
                + '\n'.join(f'- {c}' for c in unique)
            )},
        ],
        tools=[_NORMALIZE_TOOL],
        tool_choice=_tool_choice('return_normalized_concepts'),
    )
    result = _parse_tool_call(response, 'return_normalized_concepts')
    if result:
        return [c.strip().lower() for c in result.get('concepts', []) if c.strip()]
    return unique[:80]


print('Normalization function defined')

## 8 · Phase 2 — Schema Population

In [ ]:
def populate_schema_row(full_text, canonical_concepts):
    """
    For one paper's full text, extract {value, quote} for each concept.
    Returns {canonical: {'value': str, 'quote': str}}.
    """
    empty        = {c: {'value': '', 'quote': ''} for c in canonical_concepts}
    text_excerpt = (full_text or '')[:FULL_TEXT_MAX_CHARS]
    if not text_excerpt:
        return empty

    concept_list = '\n'.join(f'- {c}' for c in canonical_concepts)

    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=4096,
        messages=[
            {'role': 'system', 'content': _SCHEMA_SYSTEM},
            {'role': 'user',   'content': (
                f'Paper text (may be truncated to {FULL_TEXT_MAX_CHARS:,} characters):\n\n'
                f'{text_excerpt}\n\n'
                f'---\n'
                f'For each concept below, return the paper-specific value and source quote:\n\n'
                f'{concept_list}'
            )},
        ],
        tools=[_SCHEMA_TOOL],
        tool_choice=_tool_choice('return_schema_values'),
    )

    result = _parse_tool_call(response, 'return_schema_values')
    if not result:
        return empty

    out = empty.copy()
    for item in result.get('values', []):
        canon = item.get('canonical', '').strip().lower()
        if canon in out:
            out[canon] = {
                'value': item.get('value', '').strip(),
                'quote': item.get('quote', '').strip(),
            }
    return out


def build_schema_csv(collection_dict, canonical_concepts, domain):
    """
    Phase 2: one row per paper, one column per concept ("value | quote").
    """
    rows   = []
    papers = [p for p in collection_dict.values() if p.get('abstract') or p.get('full_text')]
    total  = len(papers)

    for i, paper in enumerate(papers, 1):
        print(f'  [{i}/{total}] Schema row: {paper["title"][:70]}')
        text        = paper.get('full_text') or paper.get('abstract', '')
        schema_data = populate_schema_row(text, canonical_concepts)

        row = {'domain': domain, 'doi': paper.get('doi', '')}
        for concept in canonical_concepts:
            cv    = schema_data.get(concept, {'value': '', 'quote': ''})
            value = cv.get('value', '')
            quote = cv.get('quote', '')
            if value and quote:
                row[concept] = f'{value} | {quote}'
            elif value:
                row[concept] = value
            elif quote:
                row[concept] = quote
            else:
                row[concept] = ''
        rows.append(row)
        if i < total:
            time.sleep(RATE_LIMIT_DELAY)

    columns = ['domain', 'doi'] + canonical_concepts
    return pd.DataFrame(rows, columns=columns)


print('Phase 2 functions defined')

## 9 · Run Workflow

Cells below execute each stage. Run them top-to-bottom, or skip to a specific stage if you already have intermediate outputs.

In [ ]:
# ── List available Zotero collections ───────────────────────────────────────
my_collections = get_collection_map(zot)
print('Available collections:')
for name, key in sorted(my_collections.items()):
    print(f'  {key}  {name}')

In [ ]:
# ── Resolve collection name and load papers ──────────────────────────────────
_id_to_name     = {v: k for k, v in my_collections.items()}
collection_name = _id_to_name.get(COLLECTION_ID, COLLECTION_ID)
domain          = collection_name.lower().replace(' ', '_')

print(f'Loading "{collection_name}" (id: {COLLECTION_ID})…')
papers = get_collection_with_text(COLLECTION_ID)
print(f'Loaded {len(papers)} papers.')

missing_pdf = [p['title'] for p in papers.values() if not p['full_text']]
if missing_pdf:
    print(f'\n{len(missing_pdf)} papers without PDF (will use abstract for Phase 2):')
    for t in missing_pdf:
        print(f'  - {t}')

In [ ]:
# ── Phase 1 or load from existing CSV ───────────────────────────────────────
if USE_CSV_CONCEPTS and CONCEPTS_CSV_PATH:
    print(f'Loading concepts from CSV: {CONCEPTS_CSV_PATH}')
    normalized_concepts = (
        pd.read_csv(CONCEPTS_CSV_PATH)[CONCEPTS_COLUMN]
        .dropna().str.strip().str.lower().tolist()
    )
    print(f'Loaded {len(normalized_concepts)} concepts.')
    df_concepts = pd.DataFrame()
else:
    print(f'[Phase 1] Extracting concepts ({TOP_N_PER_PAPER}/paper) with {MODEL}…')
    df_concepts, all_canonicals = build_concept_table(papers, top_n=TOP_N_PER_PAPER)
    print(f'\n  {len(df_concepts)} concept-paper pairs extracted.')
    print(f'  {len(set(all_canonicals))} unique raw canonical labels.')
    df_concepts.head(10)

In [ ]:
# ── Normalization ────────────────────────────────────────────────────────────
if not USE_CSV_CONCEPTS:
    print(f'[Normalization] Normalizing concept list with {MODEL}…')
    time.sleep(RATE_LIMIT_DELAY)
    normalized_concepts = normalize_concept_list(all_canonicals)
    print(f'  Normalized to {len(normalized_concepts)} concepts:')
    for c in normalized_concepts:
        print(f'    {c}')

In [ ]:
# ── Phase 2 — Schema population ─────────────────────────────────────────────
print(f'[Phase 2] Building schema ({len(normalized_concepts)} columns) with {MODEL}…')
df_schema = build_schema_csv(papers, normalized_concepts, domain)

In [ ]:
# ── Save outputs ─────────────────────────────────────────────────────────────
collection_slug = collection_name.replace(' ', '_').lower().replace('>', '').replace('-', '')
out_dir         = os.path.join('outputs', collection_slug)
os.makedirs(out_dir, exist_ok=True)
prefix = make_filename(collection_name)

if not df_concepts.empty:
    concepts_file = os.path.join(out_dir, f'concepts_{prefix}')
    df_concepts.to_csv(concepts_file, index=False)
    print(f'Saved: {concepts_file}')

schema_file = os.path.join(out_dir, f'schema_{prefix}')
df_schema.to_csv(schema_file, index=False)
print(f'Saved: {schema_file}')

schema_dir  = os.path.join('schemas', collection_slug)
os.makedirs(schema_dir, exist_ok=True)
schema_copy = os.path.join(schema_dir, f'schema_{prefix}')
df_schema.to_csv(schema_copy, index=False)
print(f'Saved: {schema_copy}')

## 10 · Results Preview

In [ ]:
# ── Concept extraction table ─────────────────────────────────────────────────
if not df_concepts.empty:
    print(f'Concept-paper pairs: {len(df_concepts)}')
    print(f'Unique canonicals  : {df_concepts["canonical"].nunique()}')
    display(df_concepts.head(20))

In [ ]:
# ── Schema table ─────────────────────────────────────────────────────────────
print(f'Schema shape: {df_schema.shape[0]} papers × {df_schema.shape[1]} columns')
print(f'\nConcept columns ({len(normalized_concepts)}):')
for c in normalized_concepts:
    print(f'  {c}')

preview_cols = ['domain', 'doi'] + normalized_concepts[:4]
available    = [c for c in preview_cols if c in df_schema.columns]
display(df_schema[available].head(5))